# LangGraph interrupts + TypeSafe

**Difficulty:** Intermediate

The companion cookbook (`cookbooks/integrations/langgraph.ipynb`) used a TypeSafe `Choice`'s `confidence` to decide *which* automated branch a support ticket takes. This one uses the same `confidence` to decide something upstream of that: *whether* to act automatically at all, or pause and wait for a person — LangGraph's `interrupt()`.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source (trimmed):</strong> docs.langchain.com/oss/javascript/langgraph/interrupts (click to expand)</summary>

# Interrupts

> Interrupts allow you to pause graph execution at specific points and wait for external input before continuing.

When an interrupt is triggered, LangGraph saves the graph state using its persistence layer and waits indefinitely until you resume execution. Interrupts work by calling `interrupt()` at any point in a graph node. The function accepts any JSON-serializable value, which is surfaced to the caller. To resume, you re-invoke the graph using `Command`, which becomes the return value of the `interrupt()` call from inside the node.

To use `interrupt()`, you need:

1. A **checkpointer** to persist the graph state.
2. A **thread ID** in your config so the runtime knows which state to resume from.
3. A call to `interrupt()` where you want to pause (payload must be JSON-serializable).

```typescript
import { interrupt } from "@langchain/langgraph";

async function approvalNode(state: State) {
  const approved = interrupt("Do you approve this action?");
  // Command({ resume: ... }) provides the value returned into this variable
  return { approved };
}
```

## Resuming

```typescript
import { Command } from "@langchain/langgraph";

const config = { configurable: { thread_id: "thread-1" } };
const result = await graph.invoke({ input: "data" }, config);
console.log(result.__interrupt__); // [{ value: 'Do you approve this action?', ... }]

await graph.invoke(new Command({ resume: true }), config);
```

You must use the **same thread ID** when resuming that was used when the interrupt occurred. The value passed to `new Command({ resume: ... })` becomes the return value of the `interrupt()` call. The node restarts from the beginning when resumed — any code before the `interrupt()` call runs again.

> `new Command({ resume: ... })` is the **only** `Command` pattern intended as input to `invoke()`. The other `Command` parameters (`update`, `goto`, `graph`) are for *returning* from node functions, not for resuming.

## Side effects called before `interrupt()` must be idempotent

Because interrupts work by re-running the node they were called from, side effects called before `interrupt()` should be idempotent. If `interrupt()` is called after an API call that updates a record, that call re-runs every time the node resumes — potentially duplicating the update. The docs' own recommended fixes: use an idempotent operation (an upsert, not an insert), place the side effect *after* the `interrupt()` call, or **separate the side effect into its own node** so it never re-runs when the interrupt's node resumes.

> The full page also covers streaming with interrupts, resuming several parallel interrupts at once, interrupts placed inside tool calls, and validating human input with a conditional edge instead of a loop — trimmed here to what this cookbook's design actually leans on. Read it at [docs.langchain.com/oss/javascript/langgraph/interrupts](https://docs.langchain.com/oss/javascript/langgraph/interrupts) any time.

</details>

## The core idea

The sibling cookbook's `classify` node already uses `confidence` to pick *which* automated branch a ticket takes. This cookbook uses the same number for a different decision: below the floor, don't take any automated action at all — pause and hand the ticket to a person, via `interrupt()`, instead of guessing.

That decision has to happen in a *different* node from the one that called TypeSafe, and this isn't just for tidiness — it's what the docs' own idempotency rule requires. `interrupt()` re-runs its node from the top every time the graph resumes. If the TypeSafe call and the `interrupt()` call lived in the same node, resuming after a human's decision would silently re-call TypeSafe too, burning a second request for an answer that never changes. Keeping `classify` (the network call) and `humanReview` (the pause) as separate nodes means resuming `humanReview` only re-runs `humanReview` — `classify`'s result is already checkpointed and never repeated.

The `interrupt()` payload itself is just the state a human needs to make the call: the ticket text, the model's guess, and the confidence that made the model's guess untrustworthy enough to ask. That payload is exactly the same shape as this repo's own confidence-gated routing story (`lessons/patterns/0007-confidence-gated-routing.ipynb`) — a human sees what the model would have done, and why it wasn't sure — just surfaced through `interrupt()`'s pause/resume mechanism instead of a synchronous branch.

In [ ]:
import { client } from "../../assets/typesafe-client.ts";
import { choice } from "npm:@typesafe-ai/sdk";
import {
  StateGraph,
  StateSchema,
  START,
  END,
  Command,
  MemorySaver,
  interrupt,
  isInterrupted,
  INTERRUPT,
} from "npm:@langchain/langgraph";
import type { GraphNode } from "npm:@langchain/langgraph";
import * as z from "npm:zod";

const CONFIDENCE_FLOOR = 0.6;

const State = new StateSchema({
  ticket: z.string(),
  category: z.string().optional(),
  confidence: z.number().optional(),
  resolution: z.string().optional(),
});

type Destination = "billing" | "account" | "feature_request" | "humanReview";

// Same seam as the sibling cookbook: one TypeSafe Choice call, routed via Command.
// Difference: below the floor, it goes to a node that actually pauses.
const classify: GraphNode<{ InputSchema: typeof State; Nodes: Destination }> = async (state) => {
  const response = await client.systemOne({
    state: state.ticket,
    questions: {
      category: choice("Determine the broad category of this support ticket", {
        billing: "Charges, invoices, refunds, subscriptions",
        account: "Login, permissions, profile, security",
        feature_request: "The user is requesting new functionality",
      }),
    },
  });
  const { category } = response.answers;
  const goto: Destination = category.confidence < CONFIDENCE_FLOOR
    ? "humanReview"
    : (category.choice as Destination);
  return new Command({
    update: { category: category.choice, confidence: category.confidence },
    goto,
  });
};

// Separate node from classify on purpose: resuming here never re-calls TypeSafe,
// because classify's own result is already checkpointed by the time this pauses.
const humanReview: GraphNode<typeof State> = (state) => {
  const decision = interrupt({
    ticket: state.ticket,
    modelGuess: state.category,
    confidence: state.confidence,
  }) as { approvedCategory: string };
  return { resolution: `human confirmed category: ${decision.approvedCategory}` };
};

const handleBilling: GraphNode<typeof State> = () => ({ resolution: "routed to billing team" });
const handleAccount: GraphNode<typeof State> = () => ({ resolution: "routed to account team" });
const handleFeature: GraphNode<typeof State> = () => ({ resolution: "logged as feature request" });

const graph = new StateGraph(State)
  .addNode("classify", classify, { ends: ["billing", "account", "feature_request", "humanReview"] })
  .addNode("billing", handleBilling)
  .addNode("account", handleAccount)
  .addNode("feature_request", handleFeature)
  .addNode("humanReview", humanReview)
  .addEdge(START, "classify")
  .addEdge("billing", END)
  .addEdge("account", END)
  .addEdge("feature_request", END)
  .addEdge("humanReview", END)
  .compile({ checkpointer: new MemorySaver() });

async function runTicket(name: string, ticket: string) {
  const config = { configurable: { thread_id: name } };
  const result = await graph.invoke({ ticket }, config);
  if (isInterrupted(result)) {
    console.log(name, "PAUSED:", result[INTERRUPT]);
    // Simulating the human's decision - in a real app this comes from a UI, hours or days later.
    const resumed = await graph.invoke(
      new Command({ resume: { approvedCategory: result.category ?? "account" } }),
      config,
    );
    console.log(name, "RESUMED:", resumed);
  } else {
    console.log(name, "AUTO:", result);
  }
}

await runTicket("clearAccountIssue", "I can't log into my account since the password reset yesterday.");
await runTicket("genuinelyVague", "Hi, I have a question about something and I'm not sure who to ask.");

## What just happened

`clearAccountIssue` is the same unambiguous ticket the sibling cookbook uses: `classify` landed on `account` at confidence 1.0, cleared the floor, and `runTicket` never saw an interrupt — straight through as `AUTO`.

`genuinelyVague` is deliberately thin on detail — no order, no login, no feature ask, just "something." It resolved to `account` at confidence 0.59, just under `CONFIDENCE_FLOOR` (0.6), and the graph actually paused: the first `invoke()` call returned with `isInterrupted(result)` true and the payload under `result[INTERRUPT]` — the ticket, the model's guess, and the confidence that made it untrustworthy. `runTicket` then simulated a human agreeing with the model's guess and resumed with the same `thread_id`, and the graph picked up exactly where it left off, landing in `humanReview` with the human's decision as its `resolution`.

Notice what *didn't* happen: TypeSafe wasn't called a second time on resume. `classify`'s `category` and `confidence` were already written into checkpointed state before the pause — resuming only re-ran `humanReview`, which is the whole reason those two calls live in separate nodes.

**Try it yourself:** predict, before running, whether `genuinelyVague` clears `0.6` or not — it's close to the floor on purpose, so a small rewording could flip it either way. Then change the simulated human's decision in `runTicket` to override the model's guess (e.g. `approvedCategory: "billing"`) instead of confirming it, and confirm the final `resolution` reflects the override, not the model's original guess.

## Related

- `cookbooks/integrations/langgraph.ipynb` — the sibling cookbook this one extends: same `classify` shape, but confidence there picks *which* automated branch instead of *whether* to pause at all
- `lessons/patterns/0007-confidence-gated-routing.ipynb` — the three-tier act/verify/don't-act idea this cookbook's `CONFIDENCE_FLOOR` implements as an actual pause instead of a same-request branch
- `research/langgraph.md` — the primary-source research this cookbook was built from, including the full "Rules of interrupts" section trimmed out of the embedded source above

Questions? Ask your agent — it has the same research this cookbook was built from, in `research/langgraph.md` and `research/typesafe-ai.md`.